# Tang et al. nCRF - Evaluation

**Learning Nonclassical Receptive Field Modulation for Contour Detection**

Evaluation notebook for the Tang nCRF model on edge detection datasets.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import cv2
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score
import json

from model import TangNet

# Setup
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# Paths
DATASET_ROOT = Path('../../datasets/HED_Small')
OUTPUT_DIR = Path('../outputs/Tang-nCRF')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset: {DATASET_ROOT}")
print(f"Output: {OUTPUT_DIR}")

## Load Model

In [ ]:
# Create model
model = TangNet().to(DEVICE)

# Load checkpoint (optional)
checkpoint_path = Path('checkpoints/best_model.pth')
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✓ Loaded checkpoint from {checkpoint_path}")
    print(f"  Epoch: {checkpoint.get('epoch', 'N/A')}")
    print(f"  Val Loss: {checkpoint.get('val_loss', 'N/A'):.4f}")
else:
    print("⚠ No checkpoint found, using untrained model")

model.eval()

# Model info
total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel parameters: {total_params:,}")

## Load Test Dataset

In [ ]:
# Load test images
test_img_dir = DATASET_ROOT / 'test' / 'images'
test_gt_dir = DATASET_ROOT / 'test' / 'edges'

test_images = sorted(
    list(test_img_dir.glob('*.jpg')) + 
    list(test_img_dir.glob('*.png'))
)

print(f"Found {len(test_images)} test images")

# Limit for quick testing (remove limit for full evaluation)
test_images = test_images[:20]
print(f"Evaluating on {len(test_images)} images")

## Run Inference

In [ ]:
predictions = []
ground_truths = []
image_names = []

with torch.no_grad():
    for img_path in tqdm(test_images, desc='Inference'):
        # Load image
        img = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Prepare input
        img_tensor = torch.from_numpy(img_rgb.transpose(2, 0, 1)).float() / 255.0
        img_tensor = img_tensor.unsqueeze(0).to(DEVICE)
        
        # Predict
        pred = model(img_tensor)
        pred = pred[0, 0].cpu().numpy()
        
        # Load ground truth
        gt_path = test_gt_dir / img_path.name.replace('.jpg', '.png')
        if not gt_path.exists():
            gt_path = test_gt_dir / img_path.name
        
        gt = cv2.imread(str(gt_path), cv2.IMREAD_GRAYSCALE)
        if gt is not None:
            gt = gt.astype(np.float32) / 255.0
        else:
            gt = np.zeros(pred.shape, dtype=np.float32)
        
        # Resize ground truth to match prediction
        if gt.shape != pred.shape:
            gt = cv2.resize(gt, (pred.shape[1], pred.shape[0]))
        
        predictions.append(pred)
        ground_truths.append(gt)
        image_names.append(img_path.name)
        
        # Save prediction
        output_path = OUTPUT_DIR / img_path.name
        cv2.imwrite(str(output_path), (pred * 255).astype(np.uint8))

print(f"\n✓ Saved {len(predictions)} predictions to {OUTPUT_DIR}")

## Calculate Metrics

In [ ]:
def calculate_metrics(predictions, ground_truths):
    """
    Calculate ODS, OIS, and AP metrics for edge detection.
    
    ODS: Optimal Dataset Scale - best F-score threshold across all images
    OIS: Optimal Image Scale - best F-score per image, then averaged
    AP: Average Precision
    """
    thresholds = np.linspace(0.05, 0.95, 30)
    
    # Per-image OIS scores
    ois_scores = []
    
    # Collect all predictions and ground truths for ODS and AP
    all_preds = []
    all_gts = []
    
    for pred, gt in zip(predictions, ground_truths):
        # Dilate ground truth for tolerance
        gt_dilated = cv2.dilate(
            (gt > 0.5).astype(np.float32),
            np.ones((3, 3))
        )
        
        # Apply Gaussian blur to prediction
        pred_smooth = cv2.GaussianBlur(pred, (3, 3), 0)
        
        # Flatten for evaluation
        pred_flat = pred_smooth.flatten()
        gt_flat = gt_dilated.flatten()
        
        # Calculate OIS (best F-score for this image)
        f_scores = []
        for thresh in thresholds:
            pred_binary = (pred_flat >= thresh).astype(float)
            
            tp = np.sum(pred_binary * gt_flat)
            fp = np.sum(pred_binary * (1 - gt_flat))
            fn = np.sum((1 - pred_binary) * gt_flat)
            
            precision = tp / (tp + fp + 1e-8)
            recall = tp / (tp + fn + 1e-8)
            f_score = 2 * precision * recall / (precision + recall + 1e-8)
            
            f_scores.append(f_score)
        
        ois_scores.append(max(f_scores))
        
        # Accumulate for ODS and AP
        all_preds.append(pred_flat)
        all_gts.append(gt_flat)
    
    # Calculate ODS (best F-score across all images)
    all_preds_concat = np.concatenate(all_preds)
    all_gts_concat = np.concatenate(all_gts)
    
    ods_scores = []
    for thresh in thresholds:
        pred_binary = (all_preds_concat >= thresh).astype(float)
        
        tp = np.sum(pred_binary * all_gts_concat)
        fp = np.sum(pred_binary * (1 - all_gts_concat))
        fn = np.sum((1 - pred_binary) * all_gts_concat)
        
        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f_score = 2 * precision * recall / (precision + recall + 1e-8)
        
        ods_scores.append((f_score, thresh))
    
    best_ods, best_thresh = max(ods_scores, key=lambda x: x[0])
    
    # Calculate AP
    if np.sum(all_gts_concat) > 0:
        ap = average_precision_score(all_gts_concat, all_preds_concat)
    else:
        ap = 0.0
    
    return {
        'ODS': float(best_ods),
        'ODS_threshold': float(best_thresh),
        'OIS': float(np.mean(ois_scores)),
        'AP': float(ap)
    }

# Calculate metrics
metrics = calculate_metrics(predictions, ground_truths)

print("\n" + "="*50)
print("Tang nCRF - Edge Detection Metrics")
print("="*50)
print(f"ODS (Optimal Dataset Scale): {metrics['ODS']:.4f}")
print(f"ODS Threshold: {metrics['ODS_threshold']:.3f}")
print(f"OIS (Optimal Image Scale): {metrics['OIS']:.4f}")
print(f"AP (Average Precision): {metrics['AP']:.4f}")
print("="*50)

# Save metrics
metrics_path = OUTPUT_DIR.parent / 'tang_ncrf_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"\n✓ Saved metrics to {metrics_path}")

## Visualize Results

In [ ]:
# Visualize random samples
num_samples = min(5, len(predictions))
indices = np.random.choice(len(predictions), num_samples, replace=False)

fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4 * num_samples))
if num_samples == 1:
    axes = axes.reshape(1, -1)

for i, idx in enumerate(indices):
    # Load original image
    img_path = test_images[idx]
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    pred = predictions[idx]
    gt = ground_truths[idx]
    
    # Resize for visualization
    if img_rgb.shape[:2] != pred.shape:
        img_rgb = cv2.resize(img_rgb, (pred.shape[1], pred.shape[0]))
        gt = cv2.resize(gt, (pred.shape[1], pred.shape[0]))
    
    # Plot
    axes[i, 0].imshow(img_rgb)
    axes[i, 0].set_title(f'Input: {image_names[idx]}')
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(gt, cmap='gray')
    axes[i, 1].set_title('Ground Truth')
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(pred, cmap='gray')
    axes[i, 2].set_title('Tang nCRF Prediction')
    axes[i, 2].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Saved visualization to {OUTPUT_DIR / 'visualization.png'}")

## Summary

The Tang nCRF model uses bio-inspired nonclassical receptive field modulation for contour detection. Key features:

1. **Center-Surround Mechanism**: Mimics contextual modulation in V1
2. **Feature Normalization**: Normalizes center and surround features before modulation
3. **Multi-scale Processing**: Three-stage architecture captures features at different scales

The model demonstrates the effectiveness of incorporating biological vision principles into deep learning architectures for edge detection tasks.